In [3]:
"""
=========================================================
TF motif scan around TSS for MULTIPLE TFs (mm10 default)
- Scans BOTH TF motifs (FOXC2 + PROX1) in one run
- Saves per-TF outputs (ALL hits + SUMMARY hits + per-gene plots + per-key plots)
- Also creates OVERLAY plots where FOXC2 + PROX1 hits are shown together

Key fixes:
- Always produces a 'gene' column in summary (avoids KeyError: 'gene')
- Robust groupby apply (handles empty groups + pandas versions)
- Scans + and - strands explicitly (both=False) to avoid weird positions
- chr-name normalization (chr1 vs 1) to reduce FASTA mismatch

Outputs (inside motif_scan_out_py/):
  per_TF/<TF>/...
  overlay_FOXC2_PROX1/...

Requirements:
  pip/conda install: pandas matplotlib pyfaidx biopython
  requests only needed if motif .jaspar is missing and you want auto-download from JASPAR
=========================================================
"""

import os
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pyfaidx import Fasta
from Bio import motifs

# requests is optional (only for downloading motif if missing)
try:
    import requests
except Exception:
    requests = None


# =========================
# 1) User settings
# =========================
PATH = '/yourDataAndCodeFolder/'
COMMON_PATH = '/yourDataAndCodeFolder/commonData/'

GENOME_BUILD = "mm10"
GTF_PATH = os.path.join(COMMON_PATH, f"{GENOME_BUILD}_genes.gtf")
FASTA_PATH = os.path.join(COMMON_PATH, f"{GENOME_BUILD}.fa")  # must be uncompressed .fa

# TFs to scan together
TF_LIST = ["PROX1", "FOXC2"]

# SUMMARY window around TSS (bp)
UP = 2500
DOWN = 0

# SCAN window around TSS (bp) for ALL hits CSV
SCAN_UP = 10000
SCAN_DOWN = 10000

# threshold per TF (you can set different values)
REL_THRESH_BY_TF = {"PROX1": 0.85, "FOXC2": 0.85}

# Output root
OUT_DIR = os.path.join(PATH, "motif_scan_out_py")
os.makedirs(OUT_DIR, exist_ok=True)

# Per-TF folder
PER_TF_DIR = os.path.join(OUT_DIR, "per_TF")
os.makedirs(PER_TF_DIR, exist_ok=True)

# Overlay folder
OVERLAY_DIR = os.path.join(OUT_DIR, f"overlay_{TF_LIST[0]}_{TF_LIST[1]}_UP{UP}_DOWN{DOWN}")
os.makedirs(OVERLAY_DIR, exist_ok=True)


# =========================
# 2) GENES_INTEREST dictionary
#    Paste your full dictionary here exactly as you have it.
# =========================
GENES_INTEREST = {
    # =======================
    # UPREGULATED in LOF
    # =======================
    "Up-in-CapLEC1-LOF_cell-cell junction assembly": ["Ace2", "Ace", "Cldn3"],
    "Up-in-CapLEC1-LOF_reverse cholesterol transport": ["Apoa1", "Apoa4"],
    "Up-in-CapLEC1-LOF_cholesterol efflux": ["Apoa1", "Apob", "Apoa4"],
    "Up-in-CapLEC1-LOF_cholesterol metabolic process": ["Apoa1", "Apob", "Apoa4", "Apoc3", "Mttp"],
    "Up-in-CapLEC1-LOF_regulation of lipoprotein lipase activity": ["Apoa1", "Apoa4", "Apoc3"],
    "Up-in-CapLEC1-LOF_triglyceride metabolic process": ["Apoa1", "Apob", "Apoa4", "Apoc3", "Pck1", "Mttp", "Dgat1"],
    "Up-in-CapLEC1-LOF_fatty acid metabolic process": ["Apoa1", "Fabp2", "Apoa4", "Cyb5a", "Gip", "Pck1", "Fabp5", "Ggt1", "Dgat1", "Asah2", "Eif6", "Hadh", "Cd74"],

    "Up-in-CapLEC2-LOF_reverse cholesterol transport": ["Apoa1", "Apoa4"],
    "Up-in-CapLEC2-LOF_cholesterol efflux": ["Apoa1", "Apoa4", "Apob", "Apoc2"],
    "Up-in-CapLEC2-LOF_cholesterol metabolic process": ["Apoa1", "Apoa4", "Apoc3", "Apob", "Mttp", "Scp2", "Cyb5r3"],
    "Up-in-CapLEC2-LOF_regulation of lipoprotein lipase activity": ["Apoa1", "Apoa4", "Apoc3", "Apoc2"],
    "Up-in-CapLEC2-LOF_fatty acid metabolic process": ["Apoa1", "Fabp2", "Apoa4", "Pck1", "Apoc2", "Scp2", "Ggt1", "Slc27a4", "Gpx4", "Asah2", "Acaa2", "Gpx1"],
    "Up-in-CapLEC2-LOF_triglyceride metabolic process": ["Apoa1", "Apoa4", "Apoc3", "Apob", "Pck1", "Apoc2", "Mttp", "Gpx1"],

    "Up-in-CapLEC3-LOF_adherens junction": ["Actb"],
    "Up-in-CapLEC3-LOF_tight junction": ["Actb"],
    "Up-in-CapLEC3-LOF_cholesterol metabolic process": ["Apoa1"],
    "Up-in-CapLEC3-LOF_cell-cell junction assembly": ["Actb"],
    "Up-in-CapLEC3-LOF_reverse cholesterol transport": ["Apoa1"],
    "Up-in-CapLEC3-LOF_fatty acid metabolic process": ["Apoa1", "Apoc2"],
    "Up-in-CapLEC3-LOF_triglyceride metabolic process": ["Apoa1", "Apoc2"],
    "Up-in-CapLEC3-LOF_cholesterol efflux": ["Apoa1", "Apoc2"],
    "Up-in-CapLEC3-LOF_regulation of lipoprotein lipase activity": ["Apoa1", "Apoc2"],

    "Up-in-ColLEC-LOF_reverse cholesterol transport": ["Apoa1", "Apoa4"],
    "Up-in-ColLEC-LOF_cholesterol metabolic process": ["Apoa1", "Apoa4", "Apoc3", "Apob"],
    "Up-in-ColLEC-LOF_fatty acid metabolic process": ["Apoa1", "Fabp2", "Apoa4", "Apoc2", "Ggt1", "Gpx4", "Fabp4", "Fabp5", "Cd36"],
    "Up-in-ColLEC-LOF_cholesterol efflux": ["Apoa1", "Apoa4", "Apob", "Apoc2"],
    "Up-in-ColLEC-LOF_triglyceride metabolic process": ["Apoa1", "Apoa4", "Apoc3", "Apob", "Apoc2", "Cd36"],
    "Up-in-ColLEC-LOF_regulation of lipoprotein lipase activity": ["Apoa1", "Apoa4", "Apoc3", "Apoc2"],

    "Up-in-ValLEC-LOF_cholesterol efflux": ["Apoa1", "Apoa4", "Apob"],
    "Up-in-ValLEC-LOF_reverse cholesterol transport": ["Apoa1", "Apoa4"],
    "Up-in-ValLEC-LOF_fatty acid metabolic process": ["Fabp1", "Fabp2", "Apoa1", "Apoa4", "Etfb", "Alox5ap", "Ltc4s", "Dbi"],
    "Up-in-ValLEC-LOF_cholesterol metabolic process": ["Apoa1", "Apoa4", "Apob", "Fdps", "Apoc3"],
    "Up-in-ValLEC-LOF_regulation of lipoprotein lipase activity": ["Apoa1", "Apoa4", "Apoc3"],
    "Up-in-ValLEC-LOF_triglyceride metabolic process": ["Apoa1", "Apoa4", "Apob", "Dbi", "Apoc3"],

    # =======================
    # DOWNREGULATED in LOF
    # =======================
    "Down-in-CapLEC1-LOF_fibrillar collagen trimer": ["Col1a1", "Col1a2", "Col3a1", "Col5a1", "Lum"],
    "Down-in-CapLEC1-LOF_banded collagen fibril": ["Col1a1", "Col1a2", "Col3a1", "Col5a1", "Lum"],
    "Down-in-CapLEC1-LOF_collagen-containing extracellular matrix": [
        "Reg3b", "Col1a1", "Col1a2", "Reg3g", "Hmcn2", "Col3a1", "Igf1", "Col6a3", "Col6a5",
        "Col6a2", "Col14a1", "Ccdc80", "Bgn", "Col6a1", "Col5a1", "Lum"
    ],
    "Down-in-CapLEC1-LOF_negative regulation of ERK1 and ERK2 cascade": ["Igf1", "Camk2n1"],
    "Down-in-CapLEC1-LOF_response to lipid": ["Defa22", "Defa21", "Ang4", "Igf1", "Defa17", "Ar", "Cfh", "Pid1", "Bmp4"],
    "Down-in-CapLEC1-LOF_negative regulation of metabolic process": [
        "Arrdc3", "Egr1", "Igfbp5", "Igf1", "Zeb2", "Ar", "Lepr", "Dapk1", "Dact1", "Pid1",
        "Fus", "Txnip", "Stc2", "Nfib", "Camk2n1", "Igfbp3", "Bmp4", "Cygb", "Celf2",
        "Tgfbr3", "Sfpq", "Kcnq1ot1"
    ],
    "Down-in-CapLEC1-LOF_collagen biosynthetic process": ["Col1a1", "Col5a1", "Bmp4", "Cygb"],

    "Down-in-CapLEC2-LOF_fibrillar collagen trimer": ["Lum", "Col1a2"],
    "Down-in-CapLEC2-LOF_banded collagen fibril": ["Lum", "Col1a2"],
    "Down-in-CapLEC2-LOF_collagen-containing extracellular matrix": [
        "Lum", "Gpc3", "Reg3b", "Col14a1", "Nav2", "Adamtsl1", "Spock2", "Col1a2"
    ],
    "Down-in-CapLEC2-LOF_response to lipid": ["Defa22", "Ang4", "Defa21", "Scd2", "Slpi", "Zfp36l1", "Defa5", "Cebpb"],
    "Down-in-CapLEC2-LOF_positive regulation of endocytosis": ["Gpc3", "Bicd1", "Wnt5a", "Mib1"],
    "Down-in-CapLEC2-LOF_negative regulation of metabolic process": [
        "Gpc3", "Slpi", "Muc16", "Zfp36l1", "Egr1", "Foxp2", "Wt1", "Atp2b4", "Tasor",
        "Hnrnpa2b1", "Fus", "Celf2", "Wnt5a", "Nfib", "Atf7ip", "Kcnq1ot1", "Txnip",
        "Srsf6", "Cebpb", "Trp53inp1", "Atrx", "Heg1"
    ],

    "Down-in-CapLEC3-LOF_negative regulation of ERK1 and ERK2 cascade": ["Spred2", "Dlg1", "Ptpn1", "Ephb2"],
    "Down-in-CapLEC3-LOF_intracellular transport": [
        "Cdk1", "Pla2g3", "Akap1", "Ap5m1", "Leprot", "Cul3", "Bag4", "Nup153", "Iws1",
        "Mia3", "Ptpn1", "Ergic2", "Thoc5", "Xpo5", "Thoc2", "Tbc1d8b", "Scp2", "Timm23",
        "Hnrnpa2b1", "Pgap1", "Tomm34", "Stxbp3", "Ift22", "Stx4a", "Cep290", "Cpsf6",
        "Ufd1", "Hspa4", "Txnip"
    ],
    "Down-in-CapLEC3-LOF_negative regulation of metabolic process": [
        "Fbh1", "Larp7", "Clock", "Nos1", "Sfswap", "Hk2", "Lin37", "Usp47", "Cdk1",
        "Smarcb1", "Pla2g3", "Spred2", "Akap1", "Birc5", "Atad2b", "Cdc45", "Cul3",
        "Znfx1", "Tiparp", "Ctnnbip1", "Rfc1", "Samd4b", "Bag4", "Rel", "Zfp692",
        "Hexim1", "Helz", "Cdyl", "Rida", "Dlg1", "Ppp2r5d", "Rgn", "Cir1", "Ptpn1",
        "Kmt2b", "Mphosph10", "Pias4", "Cbx1", "Xpo5", "Smchd1", "Zfp608", "Pfdn2",
        "Hnrnpa2b1", "Ephb2", "Ccnd1", "Fnip1", "Mybbp1a", "Eif2s1", "Ufd1", "Srsf7",
        "Nkap", "Myt1l", "Hspa4", "Zfp36l2", "Txnip", "Spink4"
    ],

    "Down-in-ColLEC-LOF_collagen-containing extracellular matrix": ["Reg3b", "Timp3", "Mgp", "Fbln5", "Eln", "Mmrn2", "Reg3g"],
    "Down-in-ColLEC-LOF_response to lipid": ["Defa22", "Ang4", "Defa21", "Klf9"],
    "Down-in-ColLEC-LOF_negative regulation of metabolic process": ["Timp3", "Fbln5", "Prox1", "Dusp1", "Ptprj", "Plat", "Fus", "Heg1", "Pecam1", "Celf1", "Klf2"],
    "Down-in-ColLEC-LOF_negative regulation of ERK1 and ERK2 cascade": ["Timp3", "Dusp1"],

    "Down-in-ValLEC-LOF_adipose tissue development": ["Cdk4", "Lncpint", "Fto", "Sirt1", "Slc25a25", "Rorc", "Ap1s2", "Adrm1"],
    # ... keep the rest of your dictionary unchanged ...
}


# =========================
# 3) Derived: ALL_GENES + mappings
# =========================
ALL_GENES = sorted({g for genes in GENES_INTEREST.values() for g in genes})

GENE2KEYS = {}
for k, genes in GENES_INTEREST.items():
    for g in genes:
        GENE2KEYS.setdefault(g, []).append(k)

def safe_filename(s: str) -> str:
    return (
        str(s)
        .replace("/", "-")
        .replace("\\", "-")
        .replace(":", "-")
        .replace("|", "-")
        .replace("\n", " ")
        .strip()
    )


# =========================
# 4) Helper functions
# =========================
def parse_attrs(attr_str: str) -> dict:
    d = {}
    for x in attr_str.strip().split(";"):
        x = x.strip()
        if not x:
            continue
        parts = x.split(" ", 1)
        if len(parts) != 2:
            continue
        k, v = parts
        d[k] = v.strip().strip('"')
    return d

def revcomp(seq: str) -> str:
    comp = str.maketrans("ACGTN", "TGCAN")
    return seq.translate(comp)[::-1]

def rel_pos(start_abs: int, tss: int, gene_strand: str) -> int:
    # upstream negative, downstream positive, strand-aware
    return (start_abs - tss) if gene_strand == "+" else (tss - start_abs)

def normalize_chrom_for_fasta(chrom: str, genome_fa: Fasta) -> str:
    if chrom in genome_fa:
        return chrom
    if chrom.startswith("chr"):
        alt = chrom.replace("chr", "", 1)
        if alt in genome_fa:
            return alt
    else:
        alt = "chr" + chrom
        if alt in genome_fa:
            return alt
    return chrom

def group_hit_str(df: pd.DataFrame, colname: str, hit_str_func) -> pd.Series:
    if df.empty:
        return pd.Series(dtype="object", name=colname)

    gb = df.groupby("gene", group_keys=False)
    try:
        out = gb.apply(hit_str_func, include_groups=False)  # newer pandas
    except TypeError:
        out = gb.apply(hit_str_func)  # older pandas

    if isinstance(out, pd.DataFrame):
        out = out.iloc[:, 0]
    out.name = colname
    return out

def ensure_gene_column(df: pd.DataFrame) -> pd.DataFrame:
    if "gene" in df.columns:
        return df
    if "index" in df.columns:
        return df.rename(columns={"index": "gene"})
    if df.index.name == "gene":
        return df.reset_index()
    return df

def download_jaspar_motif(tf_name: str, out_path: str,
                          tax_id_mouse: int = 10090,
                          tax_id_human: int = 9606,
                          collection: str = "CORE") -> dict:
    if requests is None:
        raise ImportError(
            "requests not installed, and motif .jaspar is missing.\n"
            "Install with: pip install requests\n"
            f"Or manually place: {out_path}"
        )

    def find_first_matrix_id(tax_id: int):
        q = {
            "name": tf_name,
            "collection": collection,
            "tax_id": str(tax_id),
            "version": "latest",
            "page_size": 10
        }
        r = requests.get("https://jaspar.elixir.no/api/v1/matrix/", params=q, timeout=60)
        r.raise_for_status()
        js = r.json()
        results = js.get("results", [])
        if not results:
            return None
        return results[0].get("matrix_id")

    matrix_id = find_first_matrix_id(tax_id_mouse) or find_first_matrix_id(tax_id_human)
    if not matrix_id:
        raise RuntimeError(f"No JASPAR motif found for TF={tf_name} (mouse/human fallback).")

    motif_url = f"https://jaspar.elixir.no/api/v1/matrix/{matrix_id}.jaspar"
    m = requests.get(motif_url, timeout=60)
    m.raise_for_status()

    with open(out_path, "wb") as f:
        f.write(m.content)

    return {"tf": tf_name, "matrix_id": matrix_id, "url": motif_url, "saved_as": out_path}

def load_pssm_and_threshold(tf_name: str) -> tuple:
    jaspar_path = os.path.join(COMMON_PATH, f"{tf_name}.jaspar")
    if not os.path.exists(jaspar_path):
        info = download_jaspar_motif(tf_name=tf_name, out_path=jaspar_path)
        print("Downloaded motif:", info)
    else:
        print("Motif file already exists:", jaspar_path)

    with open(jaspar_path, "r") as handle:
        motif_list = list(motifs.parse(handle, "jaspar"))
    if len(motif_list) == 0:
        raise ValueError(f"No motifs found in {jaspar_path}")

    m = motif_list[0]
    pssm = m.counts.normalize(pseudocounts=0.8).log_odds()

    rel_thr = REL_THRESH_BY_TF.get(tf_name, 0.85)
    thr = pssm.min + rel_thr * (pssm.max - pssm.min)
    return pssm, thr, jaspar_path

def scan_tf_hits(tf_name: str, pssm, thr, tx_rep: pd.DataFrame, genome: Fasta) -> pd.DataFrame:
    hits = []
    motif_len = len(pssm.alphabet)  # not reliable; use motif length from pssm keys? Better: derive from pssm.length
    motif_len = pssm.length

    for _, row in tx_rep.iterrows():
        gene = row["gene"]
        chrom_raw = row["chr"]
        gstrand = row["strand"]

        chrom = normalize_chrom_for_fasta(chrom_raw, genome)

        if gstrand == "+":
            tss = int(row["start"])
            win_start = max(1, tss - SCAN_UP)
            win_end = tss + SCAN_DOWN
        else:
            tss = int(row["end"])
            win_start = max(1, tss - SCAN_DOWN)
            win_end = tss + SCAN_UP

        if chrom not in genome:
            continue

        seq = genome[chrom][win_start - 1:win_end].upper()
        if len(seq) < motif_len:
            continue

        # '+' scan
        for pos, score in pssm.search(seq, threshold=thr, both=False):
            pos = int(pos)
            start_abs = win_start + pos
            end_abs = start_abs + motif_len - 1
            hits.append((
                tf_name, gene, chrom_raw, tss, gstrand,
                start_abs, end_abs, "+", float(score),
                int(rel_pos(start_abs, tss, gstrand))
            ))

        # '-' scan using revcomp
        seq_rc = revcomp(seq)
        for pos, score in pssm.search(seq_rc, threshold=thr, both=False):
            pos = int(pos)
            start_in_fwd = (len(seq) - pos - motif_len)
            start_abs = win_start + start_in_fwd
            end_abs = start_abs + motif_len - 1
            hits.append((
                tf_name, gene, chrom_raw, tss, gstrand,
                start_abs, end_abs, "-", float(score),
                int(rel_pos(start_abs, tss, gstrand))
            ))

    return pd.DataFrame(
        hits,
        columns=[
            "tf", "gene", "chr", "tss", "gene_strand",
            "start_abs", "end_abs", "motif_strand", "score", "start_rel"
        ]
    )

def build_summary(hits_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Returns:
      tmp_summary_window: hits filtered to [-UP, +DOWN]
      genes_with_hits_summary: per-gene summary table (only genes with >=1 hit in summary window)
    """
    if hits_df.empty:
        cols = [
            "gene",
            "n_hits_total", "n_hits_upstream", "n_hits_downstream", "n_hits_at_tss",
            "start_rel_all", "start_rel_upstream", "start_rel_downstream", "start_rel_at_tss",
            "hits_upstream", "hits_downstream", "hits_at_tss",
            "start_rel_min", "start_rel_max",
            "dict_keys"
        ]
        return hits_df.copy(), pd.DataFrame(columns=cols)

    tmp = hits_df[(hits_df["start_rel"] >= -UP) & (hits_df["start_rel"] <= DOWN)].copy()
    if tmp.empty:
        cols = [
            "gene",
            "n_hits_total", "n_hits_upstream", "n_hits_downstream", "n_hits_at_tss",
            "start_rel_all", "start_rel_upstream", "start_rel_downstream", "start_rel_at_tss",
            "hits_upstream", "hits_downstream", "hits_at_tss",
            "start_rel_min", "start_rel_max",
            "dict_keys"
        ]
        return tmp, pd.DataFrame(columns=cols)

    tmp["is_upstream"] = (tmp["start_rel"] < 0) & (tmp["start_rel"] >= -UP)
    tmp["is_downstream"] = (tmp["start_rel"] > 0) & (tmp["start_rel"] <= DOWN)
    tmp["is_at_tss"] = (tmp["start_rel"] == 0)

    def pos_str(x: pd.Series) -> str:
        vals = sorted(x.astype(int).tolist())
        return ";".join(map(str, vals))

    def hit_str(df: pd.DataFrame) -> str:
        if df.empty:
            return ""
        df2 = df.sort_values("start_rel")
        parts = []
        for _, r in df2.iterrows():
            parts.append(
                f"{int(r['start_rel'])}|{r['motif_strand']}|{float(r['score']):.3f}|"
                f"{r['chr']}:{int(r['start_abs'])}-{int(r['end_abs'])}"
            )
        return ";".join(parts)

    # counts (within summary window)
    counts_total = tmp.groupby("gene").size().rename("n_hits_total")
    counts_up = tmp[tmp["is_upstream"]].groupby("gene").size().rename("n_hits_upstream")
    counts_down = tmp[tmp["is_downstream"]].groupby("gene").size().rename("n_hits_downstream")
    counts_tss = tmp[tmp["is_at_tss"]].groupby("gene").size().rename("n_hits_at_tss")

    # relative positions
    rel_all = tmp.groupby("gene")["start_rel"].apply(pos_str).rename("start_rel_all")
    rel_up = tmp[tmp["is_upstream"]].groupby("gene")["start_rel"].apply(pos_str).rename("start_rel_upstream")
    rel_down = tmp[tmp["is_downstream"]].groupby("gene")["start_rel"].apply(pos_str).rename("start_rel_downstream")
    rel_tss = tmp[tmp["is_at_tss"]].groupby("gene")["start_rel"].apply(pos_str).rename("start_rel_at_tss")

    # detailed hit strings
    hits_up = group_hit_str(tmp[tmp["is_upstream"]], "hits_upstream", hit_str)
    hits_down = group_hit_str(tmp[tmp["is_downstream"]], "hits_downstream", hit_str)
    hits_tss = group_hit_str(tmp[tmp["is_at_tss"]], "hits_at_tss", hit_str)

    # min/max
    rel_min = tmp.groupby("gene")["start_rel"].min().astype(int).rename("start_rel_min")
    rel_max = tmp.groupby("gene")["start_rel"].max().astype(int).rename("start_rel_max")

    summary = pd.concat(
        [
            counts_total, counts_up, counts_down, counts_tss,
            rel_all, rel_up, rel_down, rel_tss,
            hits_up, hits_down, hits_tss,
            rel_min, rel_max
        ],
        axis=1
    ).reset_index()

    summary = ensure_gene_column(summary).fillna({
        "n_hits_upstream": 0,
        "n_hits_downstream": 0,
        "n_hits_at_tss": 0,
        "start_rel_upstream": "",
        "start_rel_downstream": "",
        "start_rel_at_tss": "",
        "hits_upstream": "",
        "hits_downstream": "",
        "hits_at_tss": ""
    })

    for c in ["n_hits_total", "n_hits_upstream", "n_hits_downstream", "n_hits_at_tss"]:
        if c in summary.columns:
            summary[c] = summary[c].astype(int)

    summary["dict_keys"] = summary["gene"].map(lambda g: ";".join(GENE2KEYS.get(g, [])))
    return tmp, summary

def plot_individual(tmp: pd.DataFrame, tf_name: str, out_dir: str):
    if tmp.empty:
        return
    indiv_dir = os.path.join(out_dir, f"individual_UP{UP}_DOWN{DOWN}")
    os.makedirs(indiv_dir, exist_ok=True)

    for gene in tmp["gene"].unique():
        sub_gene = tmp[tmp["gene"] == gene].copy()
        if sub_gene.empty:
            continue

        keys = GENE2KEYS.get(gene, [])
        if not keys:
            continue

        sub_plus = sub_gene[sub_gene["motif_strand"] == "+"]
        sub_minus = sub_gene[sub_gene["motif_strand"] == "-"]
        gstrand = sub_gene["gene_strand"].iloc[0]

        for key in keys:
            plt.figure(figsize=(9.5, 2.4))
            plt.axvline(0, linestyle="--")
            plt.hlines(0, -UP, DOWN, linewidth=1.0)

            if not sub_plus.empty:
                plt.scatter(sub_plus["start_rel"], [0] * len(sub_plus),
                            s=35, marker="o", alpha=0.8, label=f"{tf_name} +")
            if not sub_minus.empty:
                plt.scatter(sub_minus["start_rel"], [0] * len(sub_minus),
                            s=35, marker="^", alpha=0.8, label=f"{tf_name} -")

            plt.title(f"{gene}: {tf_name} hits (gene_strand={gstrand})\n{key}")
            plt.xlabel("Position relative to TSS (bp)  [<0 upstream, >0 downstream]")
            plt.yticks([])
            plt.xlim(-UP, DOWN)
            plt.legend(frameon=False, loc="upper right")
            plt.tight_layout()

            base = os.path.join(indiv_dir, safe_filename(f"{gene}-{tf_name}-hits_{key}"))
            plt.savefig(base + ".pdf")
            plt.savefig(base + ".svg")
            plt.close()

def plot_combined_by_key(tmp: pd.DataFrame, tf_name: str, out_dir: str):
    if tmp.empty:
        return
    comb_dir = os.path.join(out_dir, f"combined_UP{UP}_DOWN{DOWN}")
    os.makedirs(comb_dir, exist_ok=True)

    for key, genes in GENES_INTEREST.items():
        sub_key = tmp[tmp["gene"].isin(genes)].copy()
        if sub_key.empty:
            continue

        genes_with_hits = sorted(sub_key["gene"].unique())
        if not genes_with_hits:
            continue

        y_map = {g: i for i, g in enumerate(genes_with_hits)}
        sub_key["y"] = sub_key["gene"].map(y_map)

        fig_h = max(3.0, 0.35 * len(genes_with_hits) + 1.2)
        plt.figure(figsize=(11, fig_h))
        ax = plt.gca()

        ax.axvline(0, linestyle="--")
        for g, y in y_map.items():
            ax.hlines(y, -UP, DOWN, linestyles=":", linewidth=0.8, alpha=0.2)

        plus = sub_key[sub_key["motif_strand"] == "+"]
        minus = sub_key[sub_key["motif_strand"] == "-"]

        if not plus.empty:
            ax.scatter(plus["start_rel"], plus["y"], s=22, marker="o", alpha=0.8, label=f"{tf_name} +")
        if not minus.empty:
            ax.scatter(minus["start_rel"], minus["y"], s=22, marker="^", alpha=0.8, label=f"{tf_name} -")

        ax.set_yticks(list(y_map.values()))
        ax.set_yticklabels(list(y_map.keys()))
        ax.invert_yaxis()

        ax.set_xlim(-UP, DOWN)
        ax.set_xlabel("Position relative to TSS (bp)  [<0 upstream, >0 downstream]")
        ax.set_title(f"{tf_name} motif hits — {key}\nGenes shown: {len(genes_with_hits)}")
        ax.legend(frameon=False, loc="upper right")
        plt.tight_layout()

        base = os.path.join(comb_dir, safe_filename(f"{tf_name}-hits_{key}"))
        plt.savefig(base + ".pdf")
        plt.savefig(base + ".svg")
        plt.close()

def plot_overlay(tmp_by_tf: dict, overlay_out: str):
    """
    Overlay plots: show FOXC2 + PROX1 together.
    tmp_by_tf: dict(tf -> tmp summary-window hits)
    """
    os.makedirs(overlay_out, exist_ok=True)
    indiv_dir = os.path.join(overlay_out, f"individual_overlay_UP{UP}_DOWN{DOWN}")
    comb_dir = os.path.join(overlay_out, f"combined_overlay_UP{UP}_DOWN{DOWN}")
    os.makedirs(indiv_dir, exist_ok=True)
    os.makedirs(comb_dir, exist_ok=True)

    # markers per TF and strand (colors left to matplotlib defaults)
    tf_markers = {
        TF_LIST[0]: {"+": "o", "-": "^"},
        TF_LIST[1]: {"+": "s", "-": "v"},
    }

    # Collect genes that have any hit in summary window in any TF
    genes_any = set()
    for tf, tmp in tmp_by_tf.items():
        if not tmp.empty:
            genes_any.update(tmp["gene"].unique().tolist())

    if not genes_any:
        print("No overlay plots: no hits in SUMMARY window for either TF.")
        return

    # ---- Individual overlay: one per (gene, key) ----
    for gene in sorted(genes_any):
        keys = GENE2KEYS.get(gene, [])
        if not keys:
            continue

        # gene strand: take from any TF that has it
        gstrand = None
        for tf in TF_LIST:
            t = tmp_by_tf.get(tf, pd.DataFrame())
            sg = t[t["gene"] == gene]
            if not sg.empty:
                gstrand = sg["gene_strand"].iloc[0]
                break

        for key in keys:
            plt.figure(figsize=(9.8, 2.6))
            plt.axvline(0, linestyle="--")
            plt.hlines(0, -UP, DOWN, linewidth=1.0)

            plotted_any = False
            for tf in TF_LIST:
                t = tmp_by_tf.get(tf, pd.DataFrame())
                sg = t[t["gene"] == gene]
                if sg.empty:
                    continue

                for strand in ["+", "-"]:
                    ss = sg[sg["motif_strand"] == strand]
                    if ss.empty:
                        continue
                    plt.scatter(
                        ss["start_rel"], [0] * len(ss),
                        s=35, marker=tf_markers[tf][strand], alpha=0.85,
                        label=f"{tf} {strand}"
                    )
                    plotted_any = True

            if not plotted_any:
                plt.close()
                continue

            plt.title(f"{gene}: overlay hits ({TF_LIST[0]} + {TF_LIST[1]})  gene_strand={gstrand}\n{key}")
            plt.xlabel("Position relative to TSS (bp)  [<0 upstream, >0 downstream]")
            plt.yticks([])
            plt.xlim(-UP, DOWN)
            plt.legend(frameon=False, loc="upper right", ncol=2)
            plt.tight_layout()

            base = os.path.join(indiv_dir, safe_filename(f"{gene}-OVERLAY-{TF_LIST[0]}_{TF_LIST[1]}-{key}"))
            plt.savefig(base + ".pdf")
            plt.savefig(base + ".svg")
            plt.close()

    print("Overlay individual plots:", indiv_dir)

    # ---- Combined overlay per DICT_KEY ----
    for key, genes in GENES_INTEREST.items():
        # union of hits across TFs for this key
        frames = []
        for tf in TF_LIST:
            t = tmp_by_tf.get(tf, pd.DataFrame())
            if not t.empty:
                tk = t[t["gene"].isin(genes)].copy()
                if not tk.empty:
                    frames.append(tk)
        if not frames:
            continue

        sub_key = pd.concat(frames, axis=0, ignore_index=True)
        genes_with_hits = sorted(sub_key["gene"].unique())
        if not genes_with_hits:
            continue

        y_map = {g: i for i, g in enumerate(genes_with_hits)}
        sub_key["y"] = sub_key["gene"].map(y_map)

        fig_h = max(3.0, 0.35 * len(genes_with_hits) + 1.2)
        plt.figure(figsize=(12, fig_h))
        ax = plt.gca()

        ax.axvline(0, linestyle="--")
        for g, y in y_map.items():
            ax.hlines(y, -UP, DOWN, linestyles=":", linewidth=0.8, alpha=0.2)

        plotted_any = False
        for tf in TF_LIST:
            st = sub_key[sub_key["tf"] == tf]
            if st.empty:
                continue
            for strand in ["+", "-"]:
                ss = st[st["motif_strand"] == strand]
                if ss.empty:
                    continue
                ax.scatter(
                    ss["start_rel"], ss["y"],
                    s=22, marker=tf_markers[tf][strand], alpha=0.85,
                    label=f"{tf} {strand}"
                )
                plotted_any = True

        if not plotted_any:
            plt.close()
            continue

        ax.set_yticks(list(y_map.values()))
        ax.set_yticklabels(list(y_map.keys()))
        ax.invert_yaxis()
        ax.set_xlim(-UP, DOWN)
        ax.set_xlabel("Position relative to TSS (bp)  [<0 upstream, >0 downstream]")
        ax.set_title(f"OVERLAY motif hits — {key}\nTFs: {TF_LIST[0]} + {TF_LIST[1]} | Genes shown: {len(genes_with_hits)}")
        ax.legend(frameon=False, loc="upper right", ncol=2)
        plt.tight_layout()

        base = os.path.join(comb_dir, safe_filename(f"OVERLAY-{TF_LIST[0]}_{TF_LIST[1]}-hits_{key}"))
        plt.savefig(base + ".pdf")
        plt.savefig(base + ".svg")
        plt.close()

    print("Overlay combined plots:", comb_dir)


# =========================
# 5) MAIN
# =========================
genome = None
try:
    # Load genome once
    genome = Fasta(FASTA_PATH, as_raw=True, rebuild=True)

    # ---- Read GTF and pick longest transcript per gene (once) ----
    tx_rows = []
    with open(GTF_PATH, "r") as f:
        for line in f:
            if line.startswith("#"):
                continue
            parts = line.rstrip("\n").split("\t")
            if len(parts) != 9:
                continue

            chrom, source, feature, start, end, score, strand, frame, attrs = parts
            if feature not in ("transcript", "mRNA"):
                continue

            ad = parse_attrs(attrs)
            gene_name = ad.get("gene_name") or ad.get("gene") or ad.get("gene_id")
            tx_id = ad.get("transcript_id", "NA")
            if gene_name is None:
                continue

            if gene_name in ALL_GENES:
                start_i, end_i = int(start), int(end)
                tx_len = end_i - start_i + 1
                tx_rows.append((gene_name, tx_id, chrom, start_i, end_i, strand, tx_len))

    tx = pd.DataFrame(tx_rows, columns=["gene", "tx_id", "chr", "start", "end", "strand", "tx_len"])
    if tx.empty:
        raise ValueError(
            "No transcript rows found for your genes.\n"
            "Check:\n"
            "  (1) GTF has 'transcript' or 'mRNA' features\n"
            "  (2) gene symbols match your ALL_GENES list\n"
            f"  (3) GTF_PATH={GTF_PATH}"
        )

    tx_rep = tx.sort_values("tx_len", ascending=False).drop_duplicates("gene")

    missing_genes = sorted(set(ALL_GENES) - set(tx_rep["gene"].tolist()))
    if missing_genes:
        print(f"WARNING: {len(missing_genes)} genes not found in GTF transcripts (will be skipped).")

    # ---- Run each TF ----
    all_hits_by_tf = {}
    tmp_by_tf = {}

    for tf in TF_LIST:
        print("\n==============================")
        print("Scanning TF:", tf)
        print("==============================")

        tf_out = os.path.join(PER_TF_DIR, tf)
        os.makedirs(tf_out, exist_ok=True)

        pssm, thr, jaspar_path = load_pssm_and_threshold(tf)

        hits_df = scan_tf_hits(tf, pssm, thr, tx_rep, genome)
        all_hits_by_tf[tf] = hits_df

        # CSV #1: ALL hits
        hits_csv_all = os.path.join(tf_out, f"{tf}_hits_TSS_{GENOME_BUILD}_python.csv")
        hits_df.to_csv(hits_csv_all, index=False)
        print("Saved ALL hits CSV:", hits_csv_all)

        # CSV #2: SUMMARY genes with hits
        tmp, summary = build_summary(hits_df)
        tmp_by_tf[tf] = tmp

        genes_with_hits_csv = os.path.join(tf_out, f"{tf}_genes_with_hits_TSS_UP{UP}_DOWN{DOWN}_{GENOME_BUILD}_python.csv")
        summary.to_csv(genes_with_hits_csv, index=False)
        print("Saved GENES-with-hits CSV:", genes_with_hits_csv)

        # Plots
        if tmp.empty:
            print(f"No motif hits in SUMMARY window for {tf}. Try REL_THRESH_BY_TF['{tf}']=0.80 or increase UP/DOWN.")
        else:
            plot_individual(tmp, tf, tf_out)
            plot_combined_by_key(tmp, tf, tf_out)
            print("Plots saved under:", tf_out)

    # ---- Overlay outputs (FOXC2 + PROX1 together) ----
    # Save combined ALL hits CSV for both TFs
    merged_hits = pd.concat([all_hits_by_tf[tf] for tf in TF_LIST], axis=0, ignore_index=True)
    merged_all_csv = os.path.join(OVERLAY_DIR, f"{TF_LIST[0]}_{TF_LIST[1]}_ALL_hits_TSS_{GENOME_BUILD}_python.csv")
    merged_hits.to_csv(merged_all_csv, index=False)
    print("\nSaved merged ALL-hits CSV:", merged_all_csv)

    # Overlay plots
    plot_overlay(tmp_by_tf, OVERLAY_DIR)

    print("\nDONE. All outputs in:", OUT_DIR)

finally:
    if genome is not None:
        try:
            genome.close()
        except Exception:
            pass



Scanning TF: PROX1
Motif file already exists: /Users/krishangupta/Google Drive/My Drive/Desktop/Krishan_Gupta/commonData/PROX1.jaspar
Saved ALL hits CSV: /Users/krishangupta/Google Drive/My Drive/Desktop/Krishan_Gupta/Hong/temp/motif_scan_out_py/per_TF/PROX1/PROX1_hits_TSS_mm10_python.csv
Saved GENES-with-hits CSV: /Users/krishangupta/Google Drive/My Drive/Desktop/Krishan_Gupta/Hong/temp/motif_scan_out_py/per_TF/PROX1/PROX1_genes_with_hits_TSS_UP2500_DOWN0_mm10_python.csv
Plots saved under: /Users/krishangupta/Google Drive/My Drive/Desktop/Krishan_Gupta/Hong/temp/motif_scan_out_py/per_TF/PROX1

Scanning TF: FOXC2
Motif file already exists: /Users/krishangupta/Google Drive/My Drive/Desktop/Krishan_Gupta/commonData/FOXC2.jaspar
Saved ALL hits CSV: /Users/krishangupta/Google Drive/My Drive/Desktop/Krishan_Gupta/Hong/temp/motif_scan_out_py/per_TF/FOXC2/FOXC2_hits_TSS_mm10_python.csv
Saved GENES-with-hits CSV: /Users/krishangupta/Google Drive/My Drive/Desktop/Krishan_Gupta/Hong/temp/motif_

In [ ]:
# ============================================================
# FOXC2 and GATA2 motif co-occurrence in the mouse Prox1 promoter
# Region: 2,500 bp upstream of the Prox1 TSS, mm10
#
# Outputs:
#   1. Prox1_FOXC2_GATA2_motif_hits.csv
#   2. Prox1_FOXC2_GATA2_motif_overlay.pdf
#   3. Prox1_FOXC2_GATA2_motif_overlay.svg
#
# Requirements:
#   pandas, matplotlib, pyfaidx, biopython
#   requests is needed only when motif files must be downloaded
# ============================================================

import os
import pandas as pd
import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
from pyfaidx import Fasta
from Bio import motifs

try:
    import requests
except ImportError:
    requests = None


# ============================================================
# 1. User settings
# ============================================================

PATH = '/yourDataAndCodeFolder/'
COMMON_PATH = '/yourDataAndCodeFolder/commonData/'

GENOME_BUILD = "mm10"

GTF_PATH = os.path.join(COMMON_PATH, f"{GENOME_BUILD}_genes.gtf")
FASTA_PATH = os.path.join(COMMON_PATH, f"{GENOME_BUILD}.fa")

TARGET_GENE = "Prox1"
TF_LIST = ["FOXC2", "GATA2"]

UPSTREAM_BP = 2500
RELATIVE_THRESHOLD = {
    "FOXC2": 0.85,
    "GATA2": 0.85
}

OUT_DIR = os.path.join(PATH, "Prox1_FOXC2_GATA2_motif_scan")
os.makedirs(OUT_DIR, exist_ok=True)


# ============================================================
# 2. Helper functions
# ============================================================

def parse_gtf_attributes(attribute_string):
    """Convert the ninth GTF column into a dictionary."""

    attributes = {}

    for item in attribute_string.strip().split(";"):
        item = item.strip()

        if not item:
            continue

        fields = item.split(" ", 1)

        if len(fields) != 2:
            continue

        key, value = fields
        attributes[key] = value.strip().strip('"')

    return attributes


def reverse_complement(sequence):
    """Return reverse complement of a DNA sequence."""

    translation = str.maketrans("ACGTNacgtn", "TGCANtgcan")
    return sequence.translate(translation)[::-1]


def normalize_chromosome_name(chromosome, genome):
    """Match chromosome naming between GTF and FASTA."""

    if chromosome in genome:
        return chromosome

    if chromosome.startswith("chr"):
        alternative = chromosome.replace("chr", "", 1)
    else:
        alternative = "chr" + chromosome

    if alternative in genome:
        return alternative

    raise ValueError(
        f"Chromosome {chromosome} was not found in the FASTA file."
    )


def download_jaspar_motif(tf_name, output_file):
    """Download the latest available JASPAR CORE motif."""

    if requests is None:
        raise ImportError(
            f"{output_file} is missing and requests is not installed. "
            "Install requests or manually download the JASPAR motif."
        )

    query = {
        "name": tf_name,
        "collection": "CORE",
        "tax_id": "10090",
        "version": "latest",
        "page_size": 20
    }

    response = requests.get(
        "https://jaspar.elixir.no/api/v1/matrix/",
        params=query,
        timeout=60
    )
    response.raise_for_status()

    results = response.json().get("results", [])

    # Some JASPAR TF motifs may be annotated under the human taxon.
    if not results:
        query["tax_id"] = "9606"

        response = requests.get(
            "https://jaspar.elixir.no/api/v1/matrix/",
            params=query,
            timeout=60
        )
        response.raise_for_status()
        results = response.json().get("results", [])

    if not results:
        raise RuntimeError(
            f"No JASPAR motif was found for {tf_name}."
        )

    matrix_id = results[0]["matrix_id"]

    motif_url = (
        f"https://jaspar.elixir.no/api/v1/matrix/"
        f"{matrix_id}.jaspar"
    )

    motif_response = requests.get(motif_url, timeout=60)
    motif_response.raise_for_status()

    with open(output_file, "wb") as handle:
        handle.write(motif_response.content)

    print(
        f"Downloaded {tf_name}: "
        f"matrix_id={matrix_id}, file={output_file}"
    )


def load_pssm(tf_name):
    """Load a JASPAR motif and construct a PSSM."""

    jaspar_file = os.path.join(COMMON_PATH, f"{tf_name}.jaspar")

    if not os.path.exists(jaspar_file):
        download_jaspar_motif(tf_name, jaspar_file)

    with open(jaspar_file, "r") as handle:
        motif_records = list(motifs.parse(handle, "jaspar"))

    if not motif_records:
        raise ValueError(
            f"No valid motif was found in {jaspar_file}."
        )

    motif = motif_records[0]

    pssm = motif.counts.normalize(
        pseudocounts=0.8
    ).log_odds()

    relative_cutoff = RELATIVE_THRESHOLD[tf_name]

    absolute_cutoff = (
        pssm.min
        + relative_cutoff * (pssm.max - pssm.min)
    )

    return pssm, absolute_cutoff, jaspar_file


def get_prox1_transcript(gtf_path):
    """
    Retrieve the longest Prox1 transcript from the GTF file.

    The selected transcript determines the TSS used for the promoter.
    """

    transcript_rows = []

    with open(gtf_path, "r") as handle:
        for line in handle:
            if line.startswith("#"):
                continue

            fields = line.rstrip("\n").split("\t")

            if len(fields) != 9:
                continue

            (
                chromosome,
                source,
                feature,
                start,
                end,
                score,
                strand,
                frame,
                attributes
            ) = fields

            if feature not in ("transcript", "mRNA"):
                continue

            annotation = parse_gtf_attributes(attributes)

            gene_name = (
                annotation.get("gene_name")
                or annotation.get("gene")
            )

            if gene_name != TARGET_GENE:
                continue

            start = int(start)
            end = int(end)

            transcript_rows.append({
                "gene": gene_name,
                "transcript_id": annotation.get(
                    "transcript_id",
                    "NA"
                ),
                "chromosome": chromosome,
                "start": start,
                "end": end,
                "strand": strand,
                "transcript_length": end - start + 1
            })

    if not transcript_rows:
        raise ValueError(
            f"No transcript annotation was found for {TARGET_GENE} "
            f"in {gtf_path}."
        )

    transcripts = pd.DataFrame(transcript_rows)

    longest_transcript = (
        transcripts
        .sort_values("transcript_length", ascending=False)
        .iloc[0]
    )

    return longest_transcript


def extract_upstream_promoter(transcript, genome):
    """
    Extract the strand-aware 2.5-kb sequence upstream of Prox1.

    The returned sequence is oriented in the direction of
    Prox1 transcription.
    """

    chromosome_raw = transcript["chromosome"]
    chromosome = normalize_chromosome_name(
        chromosome_raw,
        genome
    )

    gene_strand = transcript["strand"]

    if gene_strand == "+":
        tss = int(transcript["start"])

        genomic_start = max(1, tss - UPSTREAM_BP)
        genomic_end = tss - 1

        promoter_sequence = str(
            genome[chromosome][genomic_start - 1:genomic_end]
        ).upper()

    else:
        tss = int(transcript["end"])

        genomic_start = tss + 1
        genomic_end = tss + UPSTREAM_BP

        genomic_sequence = str(
            genome[chromosome][genomic_start - 1:genomic_end]
        ).upper()

        promoter_sequence = reverse_complement(genomic_sequence)

    return {
        "chromosome_raw": chromosome_raw,
        "chromosome_fasta": chromosome,
        "gene_strand": gene_strand,
        "tss": tss,
        "genomic_start": genomic_start,
        "genomic_end": genomic_end,
        "sequence": promoter_sequence
    }


def relative_to_genomic_coordinates(
    promoter_position,
    motif_length,
    promoter_information
):
    """
    Convert promoter-sequence position to mm10 genomic coordinates.

    promoter_position is zero-based from the beginning of the
    transcription-oriented promoter sequence.
    """

    gene_strand = promoter_information["gene_strand"]

    if gene_strand == "+":
        genomic_start = (
            promoter_information["genomic_start"]
            + promoter_position
        )

        genomic_end = genomic_start + motif_length - 1

    else:
        genomic_end = (
            promoter_information["genomic_end"]
            - promoter_position
        )

        genomic_start = genomic_end - motif_length + 1

    return genomic_start, genomic_end


def scan_promoter_for_tf(
    tf_name,
    promoter_information,
    pssm,
    threshold
):
    """Scan both strands of the Prox1 promoter for one TF motif."""

    sequence = promoter_information["sequence"]
    motif_length = pssm.length

    hits = []

    # Scan the transcription-oriented promoter sequence.
    for position, score in pssm.search(
        sequence,
        threshold=threshold,
        both=False
    ):
        position = int(position)

        genomic_start, genomic_end = (
            relative_to_genomic_coordinates(
                position,
                motif_length,
                promoter_information
            )
        )

        # Position relative to TSS:
        # -2500 is the distal end; positions approach zero near TSS.
        relative_start = -UPSTREAM_BP + position
        relative_end = relative_start + motif_length - 1

        hits.append({
            "tf": tf_name,
            "gene": TARGET_GENE,
            "chromosome": promoter_information["chromosome_raw"],
            "tss": promoter_information["tss"],
            "gene_strand": promoter_information["gene_strand"],
            "motif_strand": "+",
            "start_rel": relative_start,
            "end_rel": relative_end,
            "start_abs": genomic_start,
            "end_abs": genomic_end,
            "score": float(score)
        })

    # Scan the reverse-complement of the promoter sequence.
    reverse_sequence = reverse_complement(sequence)

    for position, score in pssm.search(
        reverse_sequence,
        threshold=threshold,
        both=False
    ):
        position = int(position)

        forward_position = (
            len(sequence)
            - position
            - motif_length
        )

        genomic_start, genomic_end = (
            relative_to_genomic_coordinates(
                forward_position,
                motif_length,
                promoter_information
            )
        )

        relative_start = -UPSTREAM_BP + forward_position
        relative_end = relative_start + motif_length - 1

        hits.append({
            "tf": tf_name,
            "gene": TARGET_GENE,
            "chromosome": promoter_information["chromosome_raw"],
            "tss": promoter_information["tss"],
            "gene_strand": promoter_information["gene_strand"],
            "motif_strand": "-",
            "start_rel": relative_start,
            "end_rel": relative_end,
            "start_abs": genomic_start,
            "end_abs": genomic_end,
            "score": float(score)
        })

    return pd.DataFrame(hits)


def identify_overlapping_hits(hits):
    """
    Identify physical overlap between FOXC2 and GATA2 motif intervals.

    A pair is considered overlapping when their relative promoter
    intervals share at least one base pair.
    """

    foxc2_hits = hits[hits["tf"] == "FOXC2"].copy()
    gata2_hits = hits[hits["tf"] == "GATA2"].copy()

    overlap_records = []

    for foxc2_index, foxc2_row in foxc2_hits.iterrows():
        for gata2_index, gata2_row in gata2_hits.iterrows():

            overlap_start = max(
                int(foxc2_row["start_rel"]),
                int(gata2_row["start_rel"])
            )

            overlap_end = min(
                int(foxc2_row["end_rel"]),
                int(gata2_row["end_rel"])
            )

            if overlap_start <= overlap_end:
                overlap_records.append({
                    "foxc2_index": foxc2_index,
                    "gata2_index": gata2_index,
                    "overlap_start_rel": overlap_start,
                    "overlap_end_rel": overlap_end,
                    "overlap_length_bp": (
                        overlap_end - overlap_start + 1
                    )
                })

    return pd.DataFrame(overlap_records)


def plot_motif_overlay(hits, overlaps, output_prefix):
    """
    Plot FOXC2 and GATA2 motif sites in the Prox1 promoter.

    Horizontal motif bars show motif lengths.
    Black vertical regions indicate physical overlap.
    """

    fig, ax = plt.subplots(figsize=(13, 4.2))

    y_positions = {
        "FOXC2": 1,
        "GATA2": 0
    }

    marker_styles = {
        ("FOXC2", "+"): "o",
        ("FOXC2", "-"): "^",
        ("GATA2", "+"): "s",
        ("GATA2", "-"): "v"
    }

    # Promoter baseline
    for tf_name, y_value in y_positions.items():
        ax.hlines(
            y_value,
            -UPSTREAM_BP,
            0,
            linewidth=1,
            alpha=0.35
        )

    # Motif intervals and motif-start markers
    for _, row in hits.iterrows():
        tf_name = row["tf"]
        y_value = y_positions[tf_name]

        ax.hlines(
            y=y_value,
            xmin=row["start_rel"],
            xmax=row["end_rel"],
            linewidth=5,
            alpha=0.75
        )

        ax.scatter(
            row["start_rel"],
            y_value,
            marker=marker_styles[
                (tf_name, row["motif_strand"])
            ],
            s=55,
            label=(
                f"{tf_name} {row['motif_strand']} strand"
            )
        )

    # Highlight physical overlaps
    if not overlaps.empty:
        for _, overlap in overlaps.iterrows():
            ax.axvspan(
                overlap["overlap_start_rel"],
                overlap["overlap_end_rel"],
                alpha=0.25,
                hatch="//"
            )

    # TSS
    ax.axvline(
        0,
        linestyle="--",
        linewidth=1.5
    )

    ax.text(
        0,
        1.42,
        "Prox1 TSS",
        ha="right",
        va="bottom"
    )

    ax.set_xlim(-UPSTREAM_BP, 50)
    ax.set_ylim(-0.65, 1.65)

    ax.set_yticks([0, 1])
    ax.set_yticklabels(["GATA2", "FOXC2"])

    ax.set_xlabel(
        "Position relative to the Prox1 transcription start site (bp)"
    )

    ax.set_title(
        "FOXC2 and GATA2 motif sites in the mouse Prox1 promoter\n"
        "2.5-kb region upstream of the Prox1 TSS"
    )

    # Remove duplicate legend entries
    handles, labels = ax.get_legend_handles_labels()
    unique_legend = dict(zip(labels, handles))

    ax.legend(
        unique_legend.values(),
        unique_legend.keys(),
        frameon=False,
        loc="upper left",
        bbox_to_anchor=(1.01, 1)
    )

    plt.tight_layout()

    plt.savefig(
        output_prefix + ".pdf",
        bbox_inches="tight"
    )

    plt.savefig(
        output_prefix + ".svg",
        bbox_inches="tight"
    )

    plt.close(fig)


# ============================================================
# 3. Main analysis
# ============================================================

genome = None

try:
    genome = Fasta(
        FASTA_PATH,
        as_raw=True,
        rebuild=True
    )

    prox1_transcript = get_prox1_transcript(GTF_PATH)

    print("\nSelected Prox1 transcript")
    print(prox1_transcript)

    promoter_information = extract_upstream_promoter(
        prox1_transcript,
        genome
    )

    print("\nPromoter information")
    print(
        f"Chromosome: {promoter_information['chromosome_raw']}"
    )
    print(
        f"Gene strand: {promoter_information['gene_strand']}"
    )
    print(
        f"TSS: {promoter_information['tss']}"
    )
    print(
        f"Promoter genomic interval: "
        f"{promoter_information['genomic_start']}-"
        f"{promoter_information['genomic_end']}"
    )
    print(
        f"Promoter sequence length: "
        f"{len(promoter_information['sequence'])} bp"
    )

    motif_hit_tables = []

    for tf_name in TF_LIST:
        pssm, threshold, motif_file = load_pssm(tf_name)

        print(
            f"\nScanning {tf_name}; "
            f"threshold={threshold:.4f}; "
            f"motif={motif_file}"
        )

        tf_hits = scan_promoter_for_tf(
            tf_name,
            promoter_information,
            pssm,
            threshold
        )

        motif_hit_tables.append(tf_hits)

    nonempty_tables = [
        table
        for table in motif_hit_tables
        if not table.empty
    ]

    if nonempty_tables:
        all_hits = pd.concat(
            nonempty_tables,
            ignore_index=True
        )

        all_hits = all_hits.sort_values(
            ["start_rel", "tf", "motif_strand"]
        ).reset_index(drop=True)

    else:
        all_hits = pd.DataFrame(
            columns=[
                "tf",
                "gene",
                "chromosome",
                "tss",
                "gene_strand",
                "motif_strand",
                "start_rel",
                "end_rel",
                "start_abs",
                "end_abs",
                "score"
            ]
        )

    hits_output = os.path.join(
        OUT_DIR,
        "Prox1_FOXC2_GATA2_motif_hits.csv"
    )

    all_hits.to_csv(
        hits_output,
        index=False
    )

    print("\nMotif hits")
    print(all_hits)

    overlaps = identify_overlapping_hits(all_hits)

    overlap_output = os.path.join(
        OUT_DIR,
        "Prox1_FOXC2_GATA2_overlapping_hits.csv"
    )

    overlaps.to_csv(
        overlap_output,
        index=False
    )

    print("\nPhysically overlapping FOXC2–GATA2 motif pairs")
    print(overlaps)

    plot_prefix = os.path.join(
        OUT_DIR,
        "Prox1_FOXC2_GATA2_motif_overlay"
    )

    plot_motif_overlay(
        all_hits,
        overlaps,
        plot_prefix
    )

    print("\nAnalysis completed.")
    print("Motif hits:", hits_output)
    print("Overlapping hits:", overlap_output)
    print("Plot:", plot_prefix + ".pdf")
    print("Plot:", plot_prefix + ".svg")

finally:
    if genome is not None:
        try:
            genome.close()
        except Exception:
            pass